# HW2 - A GAN for paintings: build it, train it, compare it

| Part | Sections | What happens |
|---|---|---|
| I. Build | 0-8 | Your code in `losses.py`, `models/gan.py`, `train.py`, checked step by step (tests and pictures), overfit one painting and one small batch, a short run |
| II. Tools | 9-10 | Training from this notebook (settings dict, progress bars, TensorBoard) and the tools to compare and score runs |
| III. Experiments | 11-16 | Your terminal baseline, then one question per grid: L (which game?), T (longer?), R (a weaker D?), A (width?), F (the winners together, 3 seeds) |
| IV. Report | 17-18 | Every reported run in one table with test FID / IS (as HW1), and the report figures, into `report/generated/` |

`check("...")` runs the matching tests: **passed** = done, **skipped** = still a TODO, **failed** = read the message.
Thanks to `autoreload`, a saved change in a `.py` file is used at once; no kernel restart needed.

**Run All is safe (Parts II-IV).** Every run is a folder `runs/hw2/<date>-<time>-<name>/`, the same as a terminal run,
and `train_or_load` reuses the newest finished run of a name, so Run All only trains what is missing. Scores are
saved in `<run>/eval/metrics.json` and are also computed only once. Four switches in section 9: `RETRAIN = True`
trains every run again; `TRAIN_MISSING = False` only loads (never trains); `TENSORBOARD = True` starts TensorBoard for `runs/hw2`; `QUICK = True` is a smoke test of Parts II-IV (a few batches per run, saved as
`runs/hw2-quick/<date>-<time>-quick-<name>/`, never mixed with the real runs, never written into the report).

The AI policy of the course (Piazza @35): the GAN architecture, the losses, and the training round are yours.
The notebook only *checks*, *trains*, and *measures* your code; the loop around your two steps is `hw2/train.py`.

# Part I - Build the GAN step by step

In [ ]:
%load_ext autoreload
%autoreload 2

import math
import subprocess
import sys

from pathlib import Path

try:
    import dgm  # noqa: F401  (installed: the laptop, uv sync)
except ModuleNotFoundError:  # not installed (the cluster): use the repo around this notebook
    _repo = Path.cwd().resolve().parents[2]  # .../dgm (this notebook: dgm/homeworks/hw2/notebooks)
    sys.path[:0] = [str(_repo / "src"), str(_repo / "homeworks")]

import matplotlib.pyplot as plt
import torch
from torchvision.utils import make_grid

from dgm.data.wikiart import WikiArt
from dgm.utils.misc import count_parameters, get_device, seed_everything
from dgm.utils.paths import REPO_ROOT
from hw2.config import ModelConfig
from hw2.losses import generator_loss
from hw2.models import GAN, Discriminator, Generator, to_image_range, to_model_range
from hw2.train import discriminator_step, generator_step

seed_everything(0)
device = get_device()
SIZE = 64
print("device:", device)


def check(keyword):
    # Run the tests whose name contains `keyword` (same as: uv run pytest homeworks/hw2 -k keyword).
    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pytest",
            "homeworks/hw2/tests/test_gan.py",
            "-q",
            "-rs",
            "-m",
            "not slow",
            "-k",
            keyword,
        ],
        capture_output=True,
        text=True,
        cwd=REPO_ROOT,
    )
    print(result.stdout[-1500:] or result.stderr[-1500:])


def show(images, title="", nrow=8):
    # images: [N, 3, H, W] in [0, 1]
    grid = make_grid(images.detach().cpu(), nrow=nrow, padding=2, pad_value=1.0)
    plt.figure(figsize=(nrow * 1.1, grid.shape[1] / grid.shape[2] * nrow * 1.1))
    plt.imshow(grid.permute(1, 2, 0).clamp(0, 1))
    plt.title(title)
    plt.axis("off")
    plt.show()

## 0. One batch of paintings

The data loader gives `x` of shape `[B, 3, 64, 64]` with values in `[0, 1]`; the GAN works in `[-1, 1]`
(`to_model_range` in `models/gan.py`). The style labels are not used. No data yet? Then the cell uses random
pixels, and sections 1-6 still work without the 33.7 GB download.

In [ ]:
try:
    train = WikiArt("train", SIZE)
    x, styles = next(iter(torch.utils.data.DataLoader(train, batch_size=16, shuffle=True)))
    print("styles:", [train.style_names[s] for s in styles[:4]], "...")
except FileNotFoundError as err:
    print(err, "\n-> random pixels instead")
    train, x = None, torch.rand(16, 3, SIZE, SIZE)
print("x:", tuple(x.shape), f"values in [{x.min():.2f}, {x.max():.2f}]")
print(
    "to_model_range(x):",
    f"values in [{to_model_range(x).min():.2f}, {to_model_range(x).max():.2f}]",
)
show(x, "16 training paintings")

## 1. The game and its losses - `discriminator_loss`, `generator_loss` in `losses.py`

**On paper first.** With $D(x) = \sigma(l)$, where $l$ is D's logit:

$$\min_G \max_D \; V(D, G) = \mathbb E_{x \sim p_{data}}\big[\log D(x)\big] + \mathbb E_{z \sim p(z)}\big[\log(1 - D(G(z)))\big]$$

Questions for you (answer them in your own words before you code):
- For a fixed $G$, which $D^*(x)$ maximizes $V$? (Hint: maximize $p_{data}(x) \log D + p_g(x) \log(1 - D)$ for
  each $x$.) What is $V(D^*, G)$ when $p_g = p_{data}$?
- Write $\log \sigma(l)$ and $\log(1 - \sigma(l))$ with `softplus`. Why does this stay finite for $l = \pm 100$?
- Why do both losses take *logits* and not probabilities?

In [ ]:
check("dloss or gloss")

In [ ]:
# See it: G's loss and its gradient as a function of D(G(z)), for both versions of the game.
# The left end (D(G(z)) near 0) is the start of training, when D spots every fake.
try:
    logit = torch.linspace(-8, 8, 400, requires_grad=True)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
    for kind in ("minimax", "nonsaturating"):
        loss = torch.stack([generator_loss(value[None], kind) for value in logit])
        (grad,) = torch.autograd.grad(loss.sum(), logit)
        p = torch.sigmoid(logit).detach()
        axes[0].plot(p, loss.detach(), label=kind)
        axes[1].plot(p, grad.abs(), label=kind)
    axes[0].set(xlabel="D(G(z))", ylabel="generator loss")
    axes[1].set(xlabel="D(G(z))", ylabel="|d loss / d logit|", yscale="log")
    axes[0].legend()
    plt.tight_layout()
    plt.show()
except NotImplementedError as err:
    print("TODO:", err)

## 2. The generator - `Generator`

Your architecture choices go into the report: how $z$ becomes a small spatial map, how it grows to 64 x 64,
width (`g_channels`), normalization, activations. DCGAN is a good first version. An untrained generator gives
noise, of course; the point here is shapes and the value range.

In [ ]:
check("generator")
cfg = ModelConfig()
try:
    generator = Generator(cfg, SIZE)
    z = torch.randn(16, cfg.latent_dim)
    fake = generator(z)
    print(
        "generator:",
        tuple(z.shape),
        "->",
        tuple(fake.shape),
        f"values in [{fake.min():.2f}, {fake.max():.2f}] | {count_parameters(generator):,} parameters",
    )
    show(to_image_range(fake), "the UNTRAINED generator")
except NotImplementedError as err:
    print("TODO:", err)

## 3. The discriminator - `Discriminator`

One logit per image. Question for you: compare the parameter counts of D and G. What happens to G's learning
signal when D is much stronger than G?

In [ ]:
check("discriminator")
try:
    discriminator = Discriminator(cfg, SIZE)
    logits = discriminator(to_model_range(x))
    print(
        "discriminator:",
        tuple(x.shape),
        "->",
        tuple(logits.shape),
        f"| {count_parameters(discriminator):,} parameters",
    )
    print("D(x) of the untrained D:", torch.sigmoid(logits[:4]).detach().numpy().round(3))
except NotImplementedError as err:
    print("TODO:", err)

## 4. Initialization and sampling - `init_weights`, `GAN.sample`

DCGAN starts every weight from $\mathcal N(0, 0.02^2)$. The histogram shows all convolution weights after
`init_weights`. `sample` draws $z \sim \mathcal N(0, I)$ and returns images in $[0, 1]$: the images that FID scores.

Question for you: in eval mode the untrained model's samples are almost exactly gray (values near 0.5). Why?
(Hint: the weight scale 0.02, and what BatchNorm uses in eval mode before any training.)

In [ ]:
check("init or sample")
try:
    model = GAN(cfg, SIZE)
    convs = (torch.nn.Conv2d, torch.nn.ConvTranspose2d)
    weights = torch.cat(
        [m.weight.detach().flatten() for m in model.modules() if isinstance(m, convs)]
    )
    plt.hist(weights, bins=100)
    plt.title(f"convolution weights after init: std {weights.std():.4f}")
    plt.show()
    model.eval()
    samples = model.sample(16)
    print("sample:", tuple(samples.shape), f"values in [{samples.min():.2f}, {samples.max():.2f}]")
    show(samples, "GAN.sample of the UNTRAINED model (eval mode)")
except NotImplementedError as err:
    print("TODO:", err)

## 5. One round of the game - `discriminator_step`, `generator_step` in `train.py`

Algorithm 1 of the GAN paper: update D, then update G. The cell takes one step of each and counts the weight
tensors that changed. Expected: the D step changes only D, the G step changes only G.

Question for you: each network has its own optimizer. What would go wrong with ONE Adam over all parameters?

In [ ]:
check("dstep or gstep")


def changed(module, before):
    return sum(not torch.equal(p, before[name]) for name, p in module.named_parameters())


try:
    model = GAN(cfg, SIZE).to(device).train()
    opt_d = torch.optim.Adam(model.discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_g = torch.optim.Adam(model.generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
    for name in ("D step", "G step"):
        g_before = {n: p.detach().clone() for n, p in model.generator.named_parameters()}
        d_before = {n: p.detach().clone() for n, p in model.discriminator.named_parameters()}
        if name == "D step":
            terms = discriminator_step(
                model, to_model_range(x).to(device), opt_d, loss="nonsaturating"
            )
        else:
            terms = generator_step(model, len(x), opt_g, loss="nonsaturating")
        print(
            f"{name}: {terms}\n    changed tensors: G {changed(model.generator, g_before)}, "
            f"D {changed(model.discriminator, d_before)}"
        )
except NotImplementedError as err:
    print("TODO:", err)

## 6. Sanity check: a GAN on 2-D points

The classic check before images. The data are 8 Gaussian blobs on a ring. Two small MLPs (given) play the game
with YOUR steps and losses. A correct implementation moves the fake points (orange) onto the ring within a few
thousand steps (under a minute on the CPU). If nothing moves, there is a bug in the steps or the losses, not a
hyperparameter problem.

Questions for you:
- Do the fakes cover all 8 blobs, or only some? Missing blobs are **mode collapse**: G found a few outputs that
  fool D and stays there.
- Run it again with `LOSS = "minimax"` and with `LOSS = "hinge"`. What changes?

In [ ]:
LOSS = "nonsaturating"
STEPS = 3000


def ring(n, modes=8, radius=2.0, std=0.05):
    angle = 2 * torch.pi * torch.randint(modes, (n,)) / modes
    return radius * torch.stack([angle.cos(), angle.sin()], dim=1) + std * torch.randn(n, 2)


class ToyGAN(torch.nn.Module):
    # 2-D points with the GAN interface: .generator, .discriminator (-> [B] logits), .latent_dim
    def __init__(self, latent_dim=8, width=128):
        super().__init__()
        self.latent_dim = latent_dim
        nn = torch.nn
        self.generator = nn.Sequential(
            nn.Linear(latent_dim, width),
            nn.ReLU(),
            nn.Linear(width, width),
            nn.ReLU(),
            nn.Linear(width, 2),
        )
        self.discriminator = nn.Sequential(
            nn.Linear(2, width),
            nn.LeakyReLU(0.2),
            nn.Linear(width, width),
            nn.LeakyReLU(0.2),
            nn.Linear(width, 1),
            nn.Flatten(0),
        )


try:
    seed_everything(0)
    toy = ToyGAN()
    opt_d = torch.optim.Adam(toy.discriminator.parameters(), lr=1e-3, betas=(0.5, 0.999))
    opt_g = torch.optim.Adam(toy.generator.parameters(), lr=1e-3, betas=(0.5, 0.999))
    z_fixed = torch.randn(1000, toy.latent_dim)
    shots = sorted({0, STEPS // 6, STEPS // 2, STEPS})
    snapshots, history = {}, []
    for step in range(STEPS + 1):
        if step in shots:
            with torch.no_grad():
                snapshots[step] = toy.generator(z_fixed)
        if step == STEPS:
            break
        terms = discriminator_step(toy, ring(256), opt_d, loss=LOSS)
        terms |= generator_step(toy, 256, opt_g, loss=LOSS)
        history.append(terms)
    real = ring(1000)
    fig, axes = plt.subplots(1, len(snapshots) + 1, figsize=(16, 3.4))
    for ax, (step, fake) in zip(axes, snapshots.items(), strict=False):
        ax.scatter(*real.T, s=2, alpha=0.3, label="real")
        ax.scatter(*fake.T, s=2, alpha=0.5, label="fake")
        ax.set(title=f"step {step}", xlim=(-3, 3), ylim=(-3, 3), aspect="equal")
    axes[0].legend(markerscale=4)
    axes[-1].plot([h["d_real"] for h in history], lw=0.7, label="D(x)")
    axes[-1].plot([h["d_fake"] for h in history], lw=0.7, label="D(G(z))")
    axes[-1].set(xlabel="step", title=f"D outputs ({LOSS})")
    axes[-1].legend()
    plt.tight_layout()
    plt.show()
except NotImplementedError as err:
    print("TODO:", err)

## 7. Sanity checks: overfit one painting, then one small batch

As in HW1 and I2DL: before the long runs, check that the model can fit a tiny training set. A GAN has two players,
so there are two checks, one per player. Both use your losses and your two steps, on a fixed set of paintings
that never changes inside the loop.

### 7a. Overfit one painting: can G learn?

One painting, copied 16 times into every real batch. The best G then maps **every** $z$ to this painting: here
"mode collapse" is the goal. Watch the D outputs: D wins first (D(x) near 1, D(G(z)) near 0), then G catches up
and the samples become the painting. **Pass:** the mean pixel MSE between the samples and the painting falls
below 0.005 (about 750 rounds with the base settings, 20-40 s on the GPU).

Question for you: why does the non-saturating loss matter most in the first few hundred rounds here?

In [ ]:
OVERFIT_ROUNDS = 1000


def train_fixed(real, rounds, loss="nonsaturating", lr=2e-4):
    # The game on a FIXED real batch (never a new batch inside the loop). Returns the model and the history.
    seed_everything(0)
    model = GAN(cfg, SIZE).to(device).train()
    opt_d = torch.optim.Adam(model.discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_g = torch.optim.Adam(model.generator.parameters(), lr=lr, betas=(0.5, 0.999))
    history = []
    for step in range(rounds):
        terms = discriminator_step(model, real, opt_d, loss=loss)
        terms |= generator_step(model, len(real), opt_g, loss=loss)
        if not all(math.isfinite(v) for v in terms.values()):
            raise FloatingPointError(f"sanity check: non-finite value in round {step}: {terms}")
        history.append(terms)
    return model.eval(), history


def plot_game(history, title):
    # Losses and D outputs per round, as in TensorBoard (step/*).
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for key in ("d_loss", "g_loss"):
        axes[0].plot([h[key] for h in history], lw=0.7, label=key)
    axes[0].axhline(math.log(4), color="gray", ls=":", lw=0.8, label="log 4 = d_loss at D = 1/2")
    for key, label in (("d_real", "D(x)"), ("d_fake", "D(G(z))")):
        axes[1].plot([h[key] for h in history], lw=0.7, label=label)
    axes[0].set(xlabel="round", ylabel="loss", title=title)
    axes[1].set(xlabel="round", ylabel="mean D output", ylim=(0, 1), title="D outputs")
    for ax in axes:
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


def nearest(samples, paintings):
    # For each sample [N,3,H,W] in [0,1]: the pixel MSE to its nearest painting and that painting's index.
    distance = ((samples[:, None] - paintings[None]) ** 2).flatten(2).mean(2)  # [N, M]
    mse, index = distance.min(1)
    return mse, index


try:
    x_one = x[:1]  # one painting [1, 3, H, W] in [0, 1], chosen once
    real = to_model_range(x_one).repeat(16, 1, 1, 1).to(device)  # 16 copies: one real batch
    print(f"fixed batch: {tuple(real.shape)} (1 painting x 16) | loss: nonsaturating | {OVERFIT_ROUNDS} rounds")
    seed_everything(0)  # the untrained model of the same seed, for the MSE before training
    before = GAN(cfg, SIZE).eval()
    mse_before = nearest(before.sample(16), x_one)[0].mean().item()
    model, history = train_fixed(real, OVERFIT_ROUNDS)
    samples = model.sample(16).cpu()
    mse_after = nearest(samples, x_one)[0].mean().item()
    plot_game(history, "7a: one painting")
    print("last round:", {k: round(v, 3) for k, v in history[-1].items()})
    print(f"pixel MSE samples vs painting: {mse_before:.4f} -> {mse_after:.4f}  "
          f"({'PASS' if mse_after < 0.005 else 'not yet: more rounds, or check the steps'})")
    show(torch.cat([x_one, samples[:7]]), "left: the painting | then 7 new samples, z ~ N(0, I)")
except NotImplementedError as err:
    print("TODO:", err)

### 7b. Overfit one small batch: can D learn?

The 16 paintings of section 0, the same batch in every round. Now there are 16 targets, and D can memorize all
of them: D(x) goes to 1 and D(G(z)) to 0 within a few hundred rounds. **Pass:** D separates the batch from the
fakes (mean of the last 20 rounds: D(x) > 0.9 and D(G(z)) < 0.1). That checks D's learning, the counterpart of 7a
for G.

Look at what this does to G. If 7a passed, your losses and steps are fine, and still G does not copy the 16
paintings: once D wins, G's signal is weak, and G covers only a few of the paintings (**mode collapse**). The
figure shows each sample next to its nearest painting of the batch. (Measured on 15 Nov: this also fails with
hinge, with a 4x smaller D learning rate, and with 4 paintings; it is the game, not the code.) This is why a GAN
needs many paintings, and why sections 11-13 compare D(x) on training and validation paintings: a D that
memorizes the training paintings is the same failure at full scale. (Unlike the VAE in HW1, G never sees a
painting it should reproduce; G only sees D's gradient.)

In [ ]:
try:
    real = to_model_range(x).to(device)  # the 16 paintings of section 0, fixed
    print(f"fixed batch: {tuple(real.shape)} (16 paintings) | loss: nonsaturating | {OVERFIT_ROUNDS // 2} rounds")
    model, history = train_fixed(real, OVERFIT_ROUNDS // 2)
    plot_game(history, "7b: 16 paintings")
    print("last round:", {k: round(v, 3) for k, v in history[-1].items()})
    tail = history[-20:]
    end = {key: sum(h[key] for h in tail) / len(tail) for key in ("d_real", "d_fake")}
    passed = end["d_real"] > 0.9 and end["d_fake"] < 0.1
    print(f"last {len(tail)} rounds: D(x) {end['d_real']:.3f}, D(G(z)) {end['d_fake']:.3f}  "
          f"({'PASS: D can learn' if passed else 'D did not separate the batch: check the D step and the D loss'})")
    samples = model.sample(64).cpu()
    mse, index = nearest(samples, x)
    covered = torch.unique(index).numel()
    print(f"G covers {covered} of the {len(x)} paintings (nearest painting of 64 samples); "
          f"mean pixel MSE to it {mse.mean():.4f}")
    show(torch.cat([samples[:8], x[index[:8]]]), "top: 8 samples | bottom: the nearest painting of the batch")
except NotImplementedError as err:
    print("TODO:", err)

## 8. A short run on real paintings (GPU, about 2 minutes)

Only with the prepared data. `ROUNDS` rounds with the base settings, then 32 new samples. Expect color blobs with
some structure, not paintings yet. If D(x) goes to 1 and D(G(z)) to 0 within a few rounds and stays there, D wins:
check the losses, the steps, and the learning rates.

In [ ]:
ROUNDS = 300
if train is None:
    print("no data yet: uv run python -m dgm.data.wikiart --sizes 64")
else:
    try:
        seed_everything(0)
        model = GAN(cfg, SIZE).to(device).train()
        opt_d = torch.optim.Adam(model.discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
        opt_g = torch.optim.Adam(model.generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
        batch_size = min(128, len(train))
        loader = torch.utils.data.DataLoader(
            train, batch_size=batch_size, shuffle=True, drop_last=True
        )
        batches, history = iter(loader), []
        for _ in range(ROUNDS):
            real = next(batches, None)
            if real is None:
                batches = iter(loader)
                real = next(batches)
            real = to_model_range(real[0].to(device))
            terms = discriminator_step(model, real, opt_d, loss="nonsaturating")
            terms |= generator_step(model, len(real), opt_g, loss="nonsaturating")
            history.append(terms)
        fig, axes = plt.subplots(1, 2, figsize=(10, 3))
        for key in ("d_loss", "g_loss"):
            axes[0].plot([h[key] for h in history], lw=0.7, label=key)
        for key, label in (("d_real", "D(x)"), ("d_fake", "D(G(z))")):
            axes[1].plot([h[key] for h in history], lw=0.7, label=label)
        for ax in axes:
            ax.set_xlabel("round")
            ax.legend()
        plt.tight_layout()
        plt.show()
        model.eval()
        show(model.sample(32), f"after {ROUNDS} rounds")
    except NotImplementedError as err:
        print("TODO:", err)

# Part II - Tools

## 9. Training from the notebook: settings, the loop, TensorBoard

Same pipeline as in I2DL and HW1: per epoch one pass over the training set (one round of the game per batch: your
`discriminator_step`, then your `generator_step`), then one pass over the validation set (no updates). The loop is
`hw2.train.train`, the same code as `uv run python -m hw2.train`, so a notebook run and a terminal run are
the same thing and land in the same folder `runs/hw2/`.

- **Settings.** `hparams` is one flat dict, as in I2DL: every key is a field of `hw2/config.py` (data, model, or
  train). Its values are `configs/base.yaml`, i.e. your terminal baseline. A grid changes one or two keys.
- **Progress.** A bar per epoch (`Training Epoch [3/30]`, `d=` and `g=` are the running mean losses), then one line
  per epoch: D and G loss, D(x), D(G(z)), and every `fid_every` epochs the validation FID-2048.
- **TensorBoard** starts with this cell (`TENSORBOARD = True`). A TensorBoard that already serves `runs/hw2` is
  reused; otherwise the cell starts one on port 6006, or on the next free port if 6006 serves another folder.
  Open the URL the cell prints (it is also shown below the cell). By hand: `uv run tensorboard --logdir runs/hw2`.
  Every run is one entry; `step/*` are the losses per batch, `train/*` and `val/*` per epoch, and *Images* shows the
  samples of the fixed $z$ (a TensorBoard started without `--samples_per_plugin images=100` keeps only about 10
  image steps; all epochs are in `samples/epoch_XXX.png`).
- **Plots as in HW1.** `show_run(run_dir)` = `plot_history` (D and G loss train/val with the value at the balance
  point D = 1/2 of the run's loss, D outputs, validation FID per epoch) and 32 new samples of the run's `best.pt`.
- **A dead game stops early** when `stop_dead_epochs` > 0: mean D(G(z)) below $10^{-6}$ for that many epochs in a
  row (then G gets no gradient), one last validation FID, then stop. Off in `hparams`; the grid uses 2.
- **What is kept.** `checkpoints/best.pt` = the epoch with the lowest validation FID-2048 (2,000 paintings, every
  5 epochs), `last.pt`, the sample grids of a fixed $z$ (`samples/`), `metrics.csv`, `loss_curves.png`.
- **Time** (measured 15 Nov, RTX 4080 laptop, batch 128): about 40 s per epoch, so a 30-epoch run takes about
  20-25 min. Check the GPU power first: `nvidia-smi --query-gpu=power.draw,clocks.gr --format=csv`.

In [ ]:
import dataclasses
import math
import re
import socket
import time
from pathlib import Path

import numpy as np
from IPython.display import IFrame, Image, display
from omegaconf import OmegaConf

import hw2.report_assets as report_assets
import hw2.train as hw2_train
from dgm.training.checkpoint import load_checkpoint
from dgm.training.loggers import read_csv
from dgm.training.runs import create_run_dir, experiment_dir, read_json, write_json
from hw2.config import Config, DataConfig, TrainConfig
from hw2.evaluate import evaluate, load_run, print_summary

QUICK = False  # True: smoke test of Parts II-IV (a few batches per run), saved in runs/hw2-quick
RETRAIN = False  # True: train every run again, even when a finished run of that name exists
TENSORBOARD = True  # True: start TensorBoard for the runs (or reuse one that serves them) and show it here
TRAIN_MISSING = True  # False: never train, only load finished runs (missing ones are skipped)

hparams = {  # = configs/base.yaml: your terminal baseline (DCGAN recipe, Radford et al. 2016)
    # data
    "image_size": 64,
    "batch_size": 128,
    "styles": [],  # [] = all 27 styles
    # model (read by your models/gan.py)
    "latent_dim": 100,
    "g_channels": 64,
    "d_channels": 64,
    "init": "dcgan",
    # training
    "epochs": 30,
    "loss": "nonsaturating",  # "minimax", "nonsaturating", or "hinge" (your losses.py)
    "lr_g": 2e-4,
    "lr_d": 2e-4,
    "beta1": 0.5,
    "beta2": 0.999,
    "hflip": False,
    "ema_decay": 0.0,  # > 0: also keep an EMA of the weights (it does not change the training)
    "seed": 0,
    "stop_dead_epochs": 0,  # > 0: stop when D(G(z)) < 1e-6 for this many epochs in a row (a dead game)
    # monitoring: the validation FID that picks best.pt
    "fid_every": 5,
    "fid_samples": 2000,
}
# QUICK: a few small batches and tiny FID sets (the smoke test of this notebook; numbers mean nothing)
QUICK_SETTINGS = {"epochs": 1, "max_batches_per_epoch": 5, "batch_size": 8, "fid_every": 1, "fid_samples": 64}

# runs/hw2: terminal and notebook runs together. A new run moves runs/hw2/LATEST, as in the terminal, and
# section 13 points it at the chosen run. QUICK runs go to runs/hw2-quick and never touch runs/hw2.
REAL_RUNS_DIR = experiment_dir(hw2_train.EXPERIMENT)
RUNS_DIR = experiment_dir(f"{hw2_train.EXPERIMENT}-quick") if QUICK else REAL_RUNS_DIR
RUNS_DIR.mkdir(parents=True, exist_ok=True)
# Report files: the real ones go where main.tex reads them; QUICK never writes into the report.
REPORT_DIR = RUNS_DIR / "report" if QUICK else report_assets.GENERATED
REPORT_DIR.mkdir(parents=True, exist_ok=True)
# Every settings key belongs to exactly one section of hw2/config.py.
SECTION = {
    field.name: section
    for section, schema in (("data", DataConfig), ("model", ModelConfig), ("train", TrainConfig))
    for field in dataclasses.fields(schema)
}


def make_config(name, settings):
    # Flat settings dict -> the typed Config that hw2.train uses (base.yaml, then the settings on top).
    unknown = sorted(set(settings) - set(SECTION))
    if unknown:
        raise KeyError(f"unknown settings {unknown}; known: {sorted(SECTION)}")
    nested = {"name": name, "data": {}, "model": {}, "train": {}}
    for key, value in settings.items():
        nested[SECTION[key]][key] = value
    merged = OmegaConf.merge(
        OmegaConf.structured(Config), OmegaConf.load(hw2_train.DEFAULT_CONFIG), nested
    )
    return OmegaConf.to_object(merged)


def finished_runs(name, folder=None):
    # Folders <folder>/<date>-<time>-<name> of FINISHED runs (summary.json exists), oldest first.
    # A run you stopped (Interrupt / Ctrl+C) has no summary.json and is never reused.
    folder = folder or RUNS_DIR
    if not folder.exists():  # no run at all yet
        return []
    pattern = re.compile(rf"\d{{8}}-\d{{6}}-{re.escape(name)}(-\d+)?")
    return sorted(
        d for d in folder.iterdir()
        if d.is_dir() and pattern.fullmatch(d.name) and (d / "summary.json").exists()
    )


def newest_run(name):
    # The newest finished run of `name`; with QUICK the run `quick-<name>`.
    if QUICK and not name.startswith("quick-"):
        name = f"quick-{name}"
    runs = finished_runs(name)
    return runs[-1] if runs else None


def differences(run_dir, cfg):
    # Settings in which a saved run differs from `cfg` (the device does not matter).
    saved = OmegaConf.to_container(OmegaConf.load(run_dir / "config.yaml"))
    now = dataclasses.asdict(cfg)
    return sorted(
        f"{section}.{key}"
        for section in ("data", "model", "train")
        for key, value in now[section].items()
        if key != "device" and saved.get(section, {}).get(key, value) != value
    )


def train_or_load(name, settings):
    # Reuse the newest finished run of `name`, or train it (RETRAIN = True: always train).
    # A saved run with other settings gets a WARNING, so every grid describes the runs it loads.
    if QUICK:
        name, settings = f"quick-{name}", {**settings, **QUICK_SETTINGS}
    cfg = make_config(name, settings)
    run_dir = newest_run(name)
    if run_dir is not None and not RETRAIN:
        differ = differences(run_dir, cfg)
        print(f"=== {name}: saved run {run_dir.name}" + (f"  WARNING: saved with other {differ}" if differ else ""))
        return run_dir
    if not TRAIN_MISSING:
        print(f"=== {name}: no finished run here (TRAIN_MISSING = False): skipped")
        return None
    print(f"\n=== {name} ===")
    run_dir = hw2_train.train(cfg, run_dir=create_run_dir(RUNS_DIR.name, name))
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return run_dir


# The losses at the balance point D = 1/2 (all logits 0), from your losses.py: (D loss, G loss)
BALANCE = {
    "minimax": (math.log(4), -math.log(2)),
    "nonsaturating": (math.log(4), math.log(2)),
    "hinge": (2.0, 0.0),
}


def plot_history(run_dir, title=None):
    # As plot_history in HW1: the epoch curves of one run (metrics.csv), train solid, validation dashed.
    m = read_csv(run_dir / "metrics.csv")
    loss = OmegaConf.load(run_dir / "config.yaml").train.loss
    epochs, has_fid = m["epoch"], ~np.isnan(m["val/fid_2048"])
    fig, axes = plt.subplots(1, 4, figsize=(16, 3))
    for ax, key, name, ref in (
        (axes[0], "d_loss", "D loss", BALANCE[loss][0]),
        (axes[1], "g_loss", "G loss", BALANCE[loss][1]),
    ):
        ax.plot(epochs, m[f"train/{key}"], label="train")
        ax.plot(epochs, m[f"val/{key}"], "--", label="val (end of epoch)")
        ax.axhline(ref, color="gray", ls=":", lw=0.8, label=f"D = 1/2: {ref:.2f}")
        ax.set(xlabel="epoch", ylabel="loss", title=f"{name} ({loss})")
    axes[2].plot(epochs, m["train/d_real"], "C0-", label="D(x) train")
    axes[2].plot(epochs, m["val/d_real"], "C0--", label="D(x) val")
    axes[2].plot(epochs, m["train/d_fake"], "C1-", label="D(G(z)) train")
    axes[2].plot(epochs, m["val/d_fake"], "C1--", label="D(G(z)) val")
    axes[2].set(xlabel="epoch", ylabel="mean D output", ylim=(0, 1), title="D outputs")
    if has_fid.any():
        axes[3].plot(epochs[has_fid], m["val/fid_2048"][has_fid], "o-", color="C2")
        axes[3].set_yscale("log")
    axes[3].set(xlabel="epoch", ylabel="val FID-2048", title="val FID-2048 (log)")
    for ax in axes[:3]:
        ax.legend(fontsize=7)
    fig.suptitle(title or run_dir.name, fontsize=10)
    plt.tight_layout()
    plt.show()


def load_gan(run_dir, checkpoint="best"):
    # The run's GAN from checkpoints/<checkpoint>.pt (EMA weights if the run has them), eval mode.
    cfg_run = OmegaConf.to_object(
        OmegaConf.merge(OmegaConf.structured(Config), OmegaConf.load(run_dir / "config.yaml"))
    )
    if not (run_dir / "checkpoints" / f"{checkpoint}.pt").exists():
        checkpoint = "last"  # no finite validation FID: no best.pt
    cfg_run.eval.checkpoint = checkpoint
    model, weights = load_run(run_dir, cfg_run, device)
    return model.to(device).eval(), weights


def show_run(run_dir, title=None):
    # As in HW1: the curves, then 32 NEW samples of the best checkpoint.
    plot_history(run_dir, title)
    model, weights = load_gan(run_dir)
    summary = read_json(run_dir / "summary.json")
    stopped = " (stopped early: a dead game)" if summary.get("stopped_early") else ""
    seed_everything(0)
    show(model.sample(32), f"32 new samples, best.pt (epoch {summary['best_epoch']}, {weights} weights){stopped}")


def start_tensorboard(port=6006):
    # Start TensorBoard for RUNS_DIR (or reuse one that serves RUNS_DIR) and show it here.
    from tensorboard import manager

    def listening(p):
        with socket.socket() as probe:
            return probe.connect_ex(("127.0.0.1", p)) == 0

    logdir = RUNS_DIR.resolve()  # runs/ is a junction: compare real paths
    same = [
        info for info in manager.get_all()
        if info.logdir and Path(info.logdir).resolve() == logdir and listening(info.port)
    ]
    if same:
        port, status = same[0].port, "already running"
    else:
        while listening(port):  # this port serves another folder (HW1, hw2-quick, ...)
            port += 1
        log_path = RUNS_DIR / "tensorboard.log"
        log = open(log_path, "w")  # noqa: SIM115 (TensorBoard keeps writing to it)
        proc = subprocess.Popen(
            [sys.executable, "-m", "tensorboard.main", "--logdir", str(RUNS_DIR), "--port", str(port),
             "--samples_per_plugin", "images=100"],
            stdout=log,
            stderr=subprocess.STDOUT,
        )
        deadline = time.monotonic() + 60
        while not listening(port):  # the frame must not load before TensorBoard listens
            if proc.poll() is not None or time.monotonic() > deadline:
                log.flush()
                print(log_path.read_text(errors="replace")[-1500:])
                raise RuntimeError(f"TensorBoard did not start on port {port}; see {log_path}")
            time.sleep(0.5)
        status = "started"
    url = f"http://localhost:{port}"
    print(f"TensorBoard {status}: {url}  (logdir {RUNS_DIR})")
    display(IFrame(url, width="100%", height=650))


if TENSORBOARD:
    start_tensorboard()

## 10. Tools to compare and score runs

- `run_row(run_dir)` is one row of a table: the best epoch and its **validation** FID-2048 from `summary.json`
  (the number every decision uses), the epochs run, the training time, and the D columns from `d_outputs`.
- `d_outputs(run_dir)` measures D once, with the **same** raw weights of `best.pt`, on 2,000 training and 2,000
  validation paintings and on 2,000 fakes. `D(x) train` much higher than `D(x) val` = D memorizes the training
  paintings. (The per-epoch curves are not the same measurement: `train/d_real` is the mean over the epoch while
  D changes, `val/d_real` one snapshot at its end.) Cached as `d_outputs_best_n2000.json` in the run folder.
- `compare_curves(runs)` draws the validation FID-2048 and D(x), D(G(z)) of several runs in one figure.
- `score_run(run_dir)` = `hw2.evaluate`: FID-10k and IS against the **test** paintings, 3 repeats, and the
  real-data baseline (10,000 training paintings against the same test paintings). It is slow (several minutes) and
  is used once, for the chosen run only: the test split never picks a model.

In [ ]:
# QUICK: tiny score sets (at most the number of test and training paintings)
QUICK_N = min(64, len(WikiArt("test", SIZE)), len(WikiArt("train", SIZE))) if QUICK else 0
EVAL_OVERRIDES = [f"eval.num_samples={QUICK_N}", "eval.repeats=1"] if QUICK else []
D_N = 64 if QUICK else 2000  # paintings per split (and fakes) for d_outputs


@torch.no_grad()
def d_outputs(run_dir, checkpoint="best", n=D_N):
    # Mean D(x) on n training and n validation paintings and mean D(G(z)) on n fakes, all with the SAME raw
    # weights of one checkpoint, as in training (batch statistics, running averages kept). Cached.
    if not (run_dir / "checkpoints" / f"{checkpoint}.pt").exists():
        checkpoint = "last"  # no finite validation FID: no best.pt
    path = run_dir / f"d_outputs_{checkpoint}_n{n}.json"  # n in the name: a QUICK value never stands in
    if path.exists():
        return read_json(path)
    cfg_run = OmegaConf.to_object(
        OmegaConf.merge(OmegaConf.structured(Config), OmegaConf.load(run_dir / "config.yaml"))
    )
    model = hw2_train.build_model(cfg_run).to(device)
    load_checkpoint(run_dir / "checkpoints" / f"{checkpoint}.pt", model, map_location=device)  # raw, not EMA
    model.train()
    out = {"checkpoint": checkpoint}
    with hw2_train.frozen_batchnorm_stats(model):
        for split in ("train", "val"):
            data = WikiArt(split, SIZE, cfg_run.data.styles)
            pick = torch.randperm(len(data), generator=torch.Generator().manual_seed(0))[:n]
            images = data.images[pick].float() / 255
            out[split] = torch.cat(
                [torch.sigmoid(model.discriminator(to_model_range(b.to(device)))).cpu() for b in images.split(128)]
            ).mean().item()
        z = torch.randn(n, model.latent_dim, generator=torch.Generator().manual_seed(0))
        out["fake"] = torch.cat(
            [torch.sigmoid(model.discriminator(model.generator(b.to(device)))).cpu() for b in z.split(128)]
        ).mean().item()
    write_json(path, out)
    return out


def run_row(run_dir, label=None):
    cfg_run = OmegaConf.load(run_dir / "config.yaml")
    summary = read_json(run_dir / "summary.json")
    fid = summary.get("best_val_fid_2048")
    d = d_outputs(run_dir)
    return {
        "run": label or run_dir.name[16:],  # without the <date>-<time>- prefix
        "loss": cfg_run.train.loss,
        "ema": cfg_run.train.ema_decay,
        "epochs": summary.get("epochs_completed", cfg_run.train.epochs),
        "stopped": summary.get("stopped_early", False),
        "best_epoch": summary["best_epoch"],
        "val_fid": math.nan if fid is None else fid,
        "d_real_train": d["train"],
        "d_real_val": d["val"],
        "d_fake": d["fake"],
        "minutes": summary["train_time_s"] / 60,
        "dir": run_dir,
    }


def print_table(rows):
    print(
        f"{'run':<26}{'loss':<15}{'EMA':>7}{'epochs':>8}{'best ep':>9}{'val FID-2048':>14}"
        f"{'D(x) train':>12}{'D(x) val':>10}{'D(G(z))':>9}{'min':>6}"
    )
    for r in rows:
        epochs = f"{r['epochs']}{'*' if r['stopped'] else ''}"
        print(
            f"{r['run']:<26}{r['loss']:<15}{r['ema']:>7g}{epochs:>8}{r['best_epoch']:>9d}{r['val_fid']:>14.1f}"
            f"{r['d_real_train']:>12.3f}{r['d_real_val']:>10.3f}{r['d_fake']:>9.3f}{r['minutes']:>6.0f}"
        )
    if any(r["stopped"] for r in rows):
        print("* stopped early: a dead game (mean D(G(z)) < 1e-6 for stop_dead_epochs epochs)")
    print("D columns: raw weights of best.pt on 2,000 training / validation paintings and 2,000 fakes")


def grid_tex(rows):
    # The grid as a LaTeX tabular for the report (generated/grid.tex); 9 columns in 5.5 in.
    lines = [
        "% Generated by notebooks/02_gan_step_by_step.ipynb (section 14). Do not edit by hand.",
        r"\setlength{\tabcolsep}{3pt}",
        r"\begin{tabular}{@{}lllrrrrrr@{}}",
        r"\toprule",
        r"Run & Loss & EMA & Best ep. & Val FID-2048 & $D(\vx)$ train & $D(\vx)$ val & $D(G(\vz))$ & Min. \\",
        r"\midrule",
    ]
    for r in rows:
        name = r["run"].removesuffix(" (no EMA)").replace("_", r"\_")
        fid = "--" if math.isnan(r["val_fid"]) else f"{r['val_fid']:.1f}"
        ema = f"{r['ema']:g}" if r["ema"] else "none"
        best = f"{r['best_epoch']}$^\\dagger$" if r["stopped"] else str(r["best_epoch"])
        lines.append(
            rf"\texttt{{{name}}} & {r['loss']} & {ema} & {best} & {fid} & "
            rf"{r['d_real_train']:.2f} & {r['d_real_val']:.2f} & {r['d_fake']:.2f} & {r['minutes']:.0f} \\"
        )
    lines += [r"\bottomrule", r"\end{tabular}"]
    return "\n".join(lines) + "\n"


def compare_curves(runs, path=None):
    # runs: {label: run_dir}. Left: validation FID-2048 (log scale); right: D(x) and D(G(z)) per epoch.
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
    for i, (label, run_dir) in enumerate(runs.items()):
        m = read_csv(run_dir / "metrics.csv")
        has_fid = ~np.isnan(m["val/fid_2048"])
        axes[0].plot(m["epoch"][has_fid], m["val/fid_2048"][has_fid], "o-", color=f"C{i}", label=label)
        axes[1].plot(m["epoch"], m["train/d_real"], "-", color=f"C{i}", label=f"{label}: D(x)")
        axes[1].plot(m["epoch"], m["train/d_fake"], "--", color=f"C{i}", label=f"{label}: D(G(z))")
    axes[0].set(xlabel="epoch", ylabel="val FID-2048", yscale="log")
    axes[1].set(xlabel="epoch", ylabel="mean D output (train)", ylim=(0, 1))
    axes[0].legend(fontsize=8)
    axes[1].legend(fontsize=7, ncol=2)
    plt.tight_layout()
    if path is not None:
        fig.savefig(path, bbox_inches="tight")
    plt.show()


def score_run(run_dir, light=False):
    # FID / IS on the test split, as `uv run python -m hw2.evaluate --run <run>`; computed once, then read.
    # light=True: one repeat and no real-data row (about 1.5 min instead of 5); the final runs use the full protocol.
    path = run_dir / "eval" / "metrics.json"
    if path.exists() and not RETRAIN:
        results = read_json(path)
        print_summary(results)
        return results
    overrides = EVAL_OVERRIDES if QUICK else (["eval.repeats=1", "eval.real_baseline=false"] if light else [])
    return evaluate(run_dir, overrides)


def run_grid(base, changes):
    # Train or load every row of a grid (name -> one change on base); {name: run_dir} of the finished runs.
    runs = {}
    for name, change in changes.items():
        try:
            run_dir = train_or_load(name, {**base, **change})
        except FloatingPointError as err:  # NaN/Inf: the run stops, the grid goes on
            print(f"{name}: stopped, {err}")
            continue
        if run_dir is not None:
            runs[name] = run_dir
    return runs


def resolve(entry):
    # A run folder, or a run name -> its newest finished run (None if there is none here).
    return entry if isinstance(entry, Path) else newest_run(entry)


def row_of(name):
    run_dir = resolve(name)
    return None if run_dir is None else run_row(run_dir, name if isinstance(name, str) else None)


def better_than(rows, base_row, margin=0.05):
    # Names of the rows whose val FID-2048 is at least `margin` below the base row, best first.
    good = [r for r in rows if r is not None and r["val_fid"] <= (1 - margin) * base_row["val_fid"]]
    return [r["run"] for r in sorted(good, key=lambda r: r["val_fid"])]


def decide_loss(rows):
    # Section 12 rule: the lowest val FID-2048; a tie (< 5 %) with l-nonsaturating -> l-nonsaturating.
    ranked = sorted((r for r in rows if not math.isnan(r["val_fid"])), key=lambda r: r["val_fid"])
    winner = ranked[0]
    ns = next((r for r in rows if r["run"] == "l-nonsaturating"), None)
    if ns is not None and winner is not ns and ns["val_fid"] <= 1.05 * winner["val_fid"]:
        winner = ns
    return winner["run"], winner["loss"]


def seed_row(label, rows):
    # One row for one setting trained with several seeds: the mean of every number, the std in "<key>_std".
    out = {**rows[0], "run": f"{label} (mean of {len(rows)})"}
    for key in ("val_fid", "d_real_train", "d_real_val", "d_fake", "minutes", "best_epoch"):
        values = [r[key] for r in rows]
        out[key], out[f"{key}_std"] = float(np.mean(values)), float(np.std(values))
    out["best_epoch"] = round(out["best_epoch"])
    out["stopped"] = any(r["stopped"] for r in rows)
    return out


def show_grid(runs):
    # As in HW1: the curves and new samples of every run, then the last fixed-z grids side by side.
    for name, run_dir in runs.items():
        show_run(run_dir, name)
    if not runs:
        return
    fig, axes = plt.subplots(1, len(runs), figsize=(5 * len(runs), 5), squeeze=False)
    for ax, (name, run_dir) in zip(axes[0], runs.items()):
        last = sorted((run_dir / "samples").glob("epoch_*.png"))[-1]
        ax.imshow(plt.imread(last))
        ax.set_title(f"{name}, {last.stem.replace('_', ' ')}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def truncated_z(n, dim, threshold, seed):
    # Truncation trick (Brock et al. 2019): z ~ N(0, I), every coordinate with |z_i| > threshold drawn again.
    gen = torch.Generator().manual_seed(seed)
    z = torch.randn(n, dim, generator=gen)
    while (bad := z.abs() > threshold).any():
        z[bad] = torch.randn(int(bad.sum()), generator=gen)
    return z


@torch.no_grad()
def nearest_training(images, chunk=4096):
    # For each image [n, 3, H, W] in [0, 1]: the index of the nearest TRAINING painting (pixel MSE) and that MSE.
    train_images = WikiArt("train", SIZE).images
    flat = images.to(device).flatten(1)
    best_mse = torch.full((len(flat),), float("inf"), device=device)
    best_index = torch.zeros(len(flat), dtype=torch.long, device=device)
    for start in range(0, len(train_images), chunk):
        block = (train_images[start : start + chunk].to(device).float() / 255).flatten(1)
        mse = torch.cdist(flat, block).pow(2) / flat.shape[1]
        value, index = mse.min(1)
        better = value < best_mse
        best_mse[better], best_index[better] = value[better], index[better] + start
    return best_index.cpu(), best_mse.cpu(), train_images

# Part III - Experiments

As in HW1: every section asks one question with a grid, a base setting and a dict `name -> change`, one change per
row, and a decision rule written **before** the runs. Decisions use the **validation** FID-2048 only (2,000
paintings, `best.pt`); the test split is used once, in section 17, for the report.

| Section | Question | Runs | Where (suggestion) | Time |
|---|---|---|---|---|
| 11 | Your terminal baseline as it is | `BASELINE` | (done) | - |
| 12 | L: which version of the game? | `l-minimax`, `l-nonsaturating`, `l-hinge` | laptop | ~50 min |
| 13 | T: does longer training help? | `t-60ep` | cluster | ~45-60 min |
| 14 | R: does the balance of the players matter? | `r-hflip`, `r-lrd1e-4`, `r-ttur` | laptop | ~75 min |
| 15 | A: does width help? | `a-width128`, `a-dwidth32` | cluster | ~80 min |
| 16 | F: the winners together, 3 seeds | `f-final-seed0/1/2` | both | ~1-2 h |

Every grid row runs on the base `L_BASE`: the non-saturating loss + EMA 0.999 + the dead-game stop. Two machines:
run the sections of each machine, then copy the cluster's run folders into `runs/hw2` on the laptop (folder names
never clash); `TRAIN_MISSING = False` then loads everything without training.

## 11. Your terminal baseline

The run you started with `uv run python -m hw2.train` (`configs/base.yaml`: non-saturating loss, Adam 2e-4 with
$\beta = (0.5, 0.999)$, batch 128, 30 epochs, no EMA). Set `BASELINE` to its name (the part after
`<date>-<time>-`). It is only loaded here, never trained again.

Questions for the report, with your numbers:
- How do D(x) and D(G(z)) move during training? Where is the balance point of the game ($D = 1/2$,
  $\mathcal L_D = \log 4 \approx 1.39$), and how far is your run from it?
- In HW1 the negative ELBO measured the model. Why do the GAN losses not measure sample quality here?
- Do the grids in `samples/` show mode collapse (many samples alike)?

In [ ]:
BASELINE = "gan-base-local"  # the name of your terminal run
baseline_runs = finished_runs(BASELINE, REAL_RUNS_DIR)  # always the real terminal run, also with QUICK
baseline_dir = baseline_runs[-1] if baseline_runs else None
if baseline_dir is None:
    print(f"no finished run named {BASELINE!r} in {REAL_RUNS_DIR}: train it in a terminal, or set BASELINE")
else:
    print_table([run_row(baseline_dir, f"{BASELINE} (terminal)")])
    show_run(baseline_dir, f"{BASELINE} (terminal, no EMA)")

## 12. L: which version of the game trains best on WikiArt?

The three versions of `losses.py` on the baseline, one change per row; everything else stays the same (same seed,
same data order). All three keep an **EMA of the weights** (decay 0.999, Yazıcı et al. 2019). The EMA does not
change the training: it is a second copy that `best.pt` and the samples use. So `l-nonsaturating` is the terminal
baseline once more, plus the EMA: the difference between the two rows of the table is the effect of the EMA alone
(up to GPU non-determinism).

| Run | Change | What the theory predicts |
|---|---|---|
| `l-minimax` | `loss="minimax"` | G's gradient is $-\sigma(l) \to 0$ when D wins: slow or no progress at the start |
| `l-nonsaturating` | `loss="nonsaturating"` | the baseline: gradient $-(1-\sigma(l)) \to -1$ when D wins |
| `l-hinge` | `loss="hinge"` | D stops pushing at the margin $\lvert l \rvert \ge 1$; G gets a constant gradient |

**A dead game stops early** (`stop_dead_epochs = 2`, the same rule for every row): if D(G(z)) stays below
$10^{-6}$ for 2 epochs, G gets no gradient and further epochs only cost time; the run scores the validation FID
once more and stops. The table marks such a run with `*`.

**Decision rule (written before the runs):** the run with the lowest best validation FID-2048 wins; its loss goes
into section 16. Ties (less than 5 % apart): the simpler one, i.e. `l-nonsaturating`.

**Time:** about 20-25 min per run that goes all 30 epochs; a dead run stops after a few minutes. Run the cell and leave it; a stopped run is trained
again from the start next time (it has no `summary.json`).

In [ ]:
L_BASE = {**hparams, "ema_decay": 0.999, "stop_dead_epochs": 2}  # + EMA (does not change the training)
L = {  # name -> the single change on L_BASE
    "l-minimax": {"loss": "minimax"},
    "l-nonsaturating": {"loss": "nonsaturating"},
    "l-hinge": {"loss": "hinge"},
}
l_runs = run_grid(L_BASE, L)
l_rows = [run_row(run_dir, name) for name, run_dir in l_runs.items()]
context = [run_row(baseline_dir, f"{BASELINE} (no EMA)")] if baseline_dir is not None and not QUICK else []
if l_rows:
    print_table(l_rows + context)
    compare_curves(l_runs, REPORT_DIR / "grid_curves.pdf")
if len(l_rows) == len(L):
    L_WINNER, L_LOSS = decide_loss(l_rows)
    print(f"decision (section 12 rule): {L_WINNER} -> loss '{L_LOSS}' for section 16")

In [ ]:
show_grid(l_runs)

## 13. T: does longer training help?

The baseline's validation FID still fell at epoch 30 (50.9 at epoch 25, 46.1 at epoch 30). `t-60ep` is
`l-nonsaturating` with 60 epochs; everything else, including the seed, is the same, so its first 30 epochs repeat
`l-nonsaturating` (up to GPU non-determinism). `best.pt` can come from any epoch.

**Decision rule (written before the run):** 60 epochs go into section 16 if `t-60ep` beats `l-nonsaturating` by at
least 5 % in best validation FID-2048; otherwise 30.

Questions for the report: where does the FID curve flatten? Does D memorize more with longer training (the D
columns: training vs validation paintings)?

In [ ]:
T_BASE = L_BASE  # the non-saturating loss + EMA + the dead-game stop
T = {"t-60ep": {"epochs": 60}}
t_runs = run_grid(T_BASE, T)
ns_row = row_of("l-nonsaturating")
t_rows = [run_row(run_dir, name) for name, run_dir in t_runs.items()]
if t_rows:
    print_table(t_rows + ([ns_row] if ns_row else []))
    compare_curves({**({"l-nonsaturating": ns_row["dir"]} if ns_row else {}), **t_runs})
    show_grid(t_runs)

## 14. R: does the balance of the players matter?

7b showed what a D that wins does to G, and a D that memorizes the training paintings gives G no useful signal
for new ones. (Your baseline: D(x) 0.93 on training vs about 0.85 on validation paintings, same checkpoint, a
small gap.) Three changes on `L_BASE`, one per row:

| Run | Change | Hypothesis |
|---|---|---|
| `r-hflip` | mirrored training paintings at random | twice the distinct paintings for D: less memorization |
| `r-lrd1e-4` | D learning rate 1e-4 (G stays 2e-4) | a slower D wins less often; G keeps a useful gradient |
| `r-ttur` | D 4e-4, G 1e-4 (Heusel et al. 2017) | the opposite: a fast D that stays close to its optimum $D^*$, so G follows a good gradient |

(Spectral normalization of D, Miyato et al. 2018, is the classic fix too, but it changes your `Discriminator`: add it
there and as a row here if you want it.)

**Decision rule (written before the runs):** the single best row goes into section 16 if it beats
`l-nonsaturating` by at least 5 % in best validation FID-2048; otherwise none.

In [ ]:
R_BASE = L_BASE
R = {  # name -> the single change on R_BASE
    "r-hflip": {"hflip": True},
    "r-lrd1e-4": {"lr_d": 1e-4},
    "r-ttur": {"lr_d": 4e-4, "lr_g": 1e-4},
}
r_runs = run_grid(R_BASE, R)
ns_row = row_of("l-nonsaturating")  # the base row (None if it is not on this machine)
r_rows = [run_row(run_dir, name) for name, run_dir in r_runs.items()]
if r_rows:
    print_table(r_rows + ([ns_row] if ns_row else []))
    compare_curves({**({"l-nonsaturating": ns_row["dir"]} if ns_row else {}), **r_runs})
    show_grid(r_runs)

## 15. A: does width help?

Your architecture stays as you wrote it; only the widths change (`g_channels`, `d_channels` in your
`models/gan.py`). Activation functions and depth are not varied: ReLU in G / LeakyReLU in D is the tested DCGAN
rule, and 64 x 64 already has the DCGAN depth.

| Run | Change | Hypothesis |
|---|---|---|
| `a-width128` | G and D twice as wide | more capacity for 27 styles (about 4x the parameters, 2-3x the time) |
| `a-dwidth32` | D half as wide | a weaker D memorizes less (the R question from the other side) |

**Decision rule (written before the runs):** as in section 14: the best row goes into section 16 if it beats
`l-nonsaturating` by at least 5 %; otherwise none.

In [ ]:
A_BASE = L_BASE
A = {  # name -> the single change on A_BASE
    "a-width128": {"g_channels": 128, "d_channels": 128},
    "a-dwidth32": {"d_channels": 32},
}
a_runs = run_grid(A_BASE, A)
ns_row = row_of("l-nonsaturating")  # the base row (None if it is not on this machine)
a_rows = [run_row(run_dir, name) for name, run_dir in a_runs.items()]
if a_rows:
    print_table(a_rows + ([ns_row] if ns_row else []))
    compare_curves({**({"l-nonsaturating": ns_row["dir"]} if ns_row else {}), **a_runs})
    show_grid(a_runs)

## 16. F: the winners together, 3 seeds

Sections 12-15 decided one thing each, every one on `L_BASE`. Here the winners go together once, with error bars
(seeds 0, 1, 2), as section 14 of HW1. The rule, written before: the loss of section 12, the best change of R and of
A if it won its section, and 60 epochs if section 13 says so. Nothing else changes. One-at-a-time winners do not
always add up (HW1, section 14), so the table puts the final setting next to `l-nonsaturating`.

**Two machines.** This cell needs all grid runs (or the file `runs/hw2/f_setting.json`, which it writes once it has
them). To train seed 2 on the cluster: copy `f_setting.json` into `~/dgm/runs/hw2/` there, set `F_TRAIN_HERE = [2]`
there and `F_TRAIN_HERE = [0, 1]` here.

In [ ]:
F_SEEDS = [0, 1, 2]
F_TRAIN_HERE = F_SEEDS  # the seeds this machine trains; the others are only loaded
F_FILE = RUNS_DIR / "f_setting.json"
ns_row = row_of("l-nonsaturating")
grids_here = all(grid in globals() for grid in ("L", "T", "R", "A"))  # the grid cells ran on this machine
rows = {name: row_of(name) for name in [*L, *T, *R, *A]} if grids_here else {}
missing = [name for name, row in rows.items() if row is None] if grids_here else ["(grid cells 12-15 not run)"]
if not missing:
    ns_row = rows["l-nonsaturating"]
    _, loss = decide_loss([rows[name] for name in L])
    change, decisions = {"loss": loss}, {"loss": loss}
    for section, grid in (("R", R), ("A", A)):
        wins = better_than([rows[name] for name in grid], ns_row)
        decisions[section] = wins[0] if wins else None
        if wins:
            change.update(grid[wins[0]])
    decisions["epochs"] = 60 if better_than([rows["t-60ep"]], ns_row) else 30
    change["epochs"] = decisions["epochs"]
    F_SETTING = {**L_BASE, **change}
    write_json(F_FILE, {"decisions": decisions, "settings": F_SETTING})
elif F_FILE.exists():
    saved = read_json(F_FILE)
    decisions, F_SETTING = saved["decisions"], saved["settings"]
    print(f"grid runs missing here ({missing}): using {F_FILE.name} from the other machine")
else:
    raise RuntimeError(f"section 16 needs the runs {missing}, or {F_FILE} from the machine that has them")
print("F decisions:", decisions)

f_runs = {}
for seed in F_SEEDS:
    name = f"f-final-seed{seed}"
    run_dir = train_or_load(name, {**F_SETTING, "seed": seed}) if seed in F_TRAIN_HERE else newest_run(name)
    if run_dir is not None:
        f_runs[name] = run_dir
f_rows = [run_row(run_dir, name) for name, run_dir in f_runs.items()]
if f_rows:
    summary_rows = [seed_row("final", f_rows)] if len(f_rows) > 1 else []
    print_table(f_rows + summary_rows + ([ns_row] if ns_row else []))
    show_grid(f_runs)

# Part IV - Into the report

## 17. Every reported run in one table

As section 17 of HW1. `REPORT` lists the rows of the report table: a label and its runs (several runs = the seeds
of one setting: mean $\pm$ std over the seeds). Every run is scored on the **test** paintings the same way:
FID-10k (10,000 samples against 10,000 test paintings) and IS; the final seeds use the full protocol (3 repeats and
the real-data row; the validation column gets the same floor at 2,000 paintings), the other rows one repeat (about 1.5 min each instead of 5). The test scores are only
reported: no decision uses them. The cell also points `runs/hw2/LATEST` at `f-final-seed0` and exports it with
`hw2.report_assets` (curves, samples, interpolation, hyperparameters, `\Result...` macros).

Writes `report/generated/results_table.tex` (read by `report/main.tex`).

In [ ]:
REPORT = {  # label -> its runs (names or folders); several runs = the seeds of one setting
    f"{BASELINE} (no EMA)": [baseline_dir] if baseline_dir is not None and not QUICK else [],
    **{name: [name] for name in [*L, *T, *R, *A]},
    "final": [f"f-final-seed{seed}" for seed in F_SEEDS],
}
final_dirs = [d for d in map(resolve, REPORT["final"]) if d is not None]
if not final_dirs:
    raise RuntimeError("no f-final run here yet: run section 16 first")
report_rows = {}
for label, entries in REPORT.items():
    run_dirs = [d for d in map(resolve, entries) if d is not None]
    if not run_dirs:
        print(f"{label}: no finished run here, left out")
        continue
    rows = []
    for run_dir in run_dirs:
        scores = score_run(run_dir, light=label != "final")
        row = run_row(run_dir, label)
        for key in ("fid_2048", "fid", "inception_score"):
            row[f"test_{key}"] = scores["samples"][key]["mean"]
        rows.append(row)
    row = seed_row(label, rows) if len(rows) > 1 else rows[0]
    for key in ("test_fid_2048", "test_fid", "test_inception_score"):
        values = [r[key] for r in rows]
        row[key] = float(np.mean(values))
        if len(rows) > 1:
            row[f"{key}_std"] = float(np.std(values))
    row["run"] = label
    report_rows[label] = row
real = read_json(final_dirs[0] / "eval" / "metrics.json").get("real_train_images", {})
# The floor of the validation FID: 2,000 training paintings against the same 2,000 validation paintings the
# training loop scores (seed 0). Computed once (about 1 min), cached next to the runs.
REAL_VAL_FILE = RUNS_DIR / f"real_val_fid_n{L_BASE['fid_samples'] if not QUICK else QUICK_SETTINGS['fid_samples']}.json"
if not REAL_VAL_FILE.exists():
    from dgm.evaluation.metrics import InceptionScorer

    n_val = QUICK_SETTINGS["fid_samples"] if QUICK else L_BASE["fid_samples"]
    val_real = hw2_train.random_subset(WikiArt("val", SIZE).as_float(), n_val, 0)
    train_real = hw2_train.random_subset(WikiArt("train", SIZE).as_float(), len(val_real), 0)
    write_json(REAL_VAL_FILE, InceptionScorer(device)(train_real, val_real))
real_val = read_json(REAL_VAL_FILE)

print(f"{'run':<26}{'ep':>4}{'val FID':>9}{'test FID-2048':>18}{'FID tut.':>10}{'IS':>14}{'D(x) tr/val':>13}")
for label, r in report_rows.items():
    test = f"{r['test_fid_2048']:.1f}" + (f" +- {r['test_fid_2048_std']:.1f}" if "test_fid_2048_std" in r else "")
    is_ = f"{r['test_inception_score']:.2f}" + (f" +- {r['test_inception_score_std']:.2f}" if "test_inception_score_std" in r else "")
    print(f"{label:<26}{r['epochs']:>4}{r['val_fid']:>9.1f}{test:>18}{r['test_fid']:>10.4f}{is_:>14}"
          f"{r['d_real_train']:>7.2f}/{r['d_real_val']:.2f}")
if real:
    print(f"{'real paintings (train)':<26}{'':>4}{real_val['fid_2048']:>9.1f}{real['fid_2048']:>18.1f}{real['fid']:>10.4f}{real['inception_score']:>14.2f}")


def pm(row, key, digits):
    text = f"{row[key]:.{digits}f}"
    return text + (f"$\\pm${row[key + '_std']:.{digits}f}" if f"{key}_std" in row else "")


lines = [
    "% Generated by notebooks/02_gan_step_by_step.ipynb (section 17). Do not edit by hand.",
    r"\setlength{\tabcolsep}{3pt}",
    r"\begin{tabular}{@{}lrrrrrr@{}}",
    r"\toprule",
    r" & & \multicolumn{1}{c}{Val} & \multicolumn{3}{c}{Test (FID-10k)} & \\",
    r"\cmidrule(lr){3-3}\cmidrule(lr){4-6}",
    r"Run & Ep. & FID$_{2048}$ & FID$_{2048}$ & FID$_{\mathrm{tut}}$ & IS & $D(\vx)$ tr./val \\",
    r"\midrule",
]
for label, r in report_rows.items():
    name = label.removesuffix(" (no EMA)").replace("_", r"\_")
    epochs = f"{r['epochs']}$^\\dagger$" if r["stopped"] else str(r["epochs"])
    lines.append(
        rf"\texttt{{{name}}} & {epochs} & {pm(r, 'val_fid', 1)} & {pm(r, 'test_fid_2048', 1)} & "
        rf"{r['test_fid']:.3f} & {pm(r, 'test_inception_score', 2)} & {r['d_real_train']:.2f}/{r['d_real_val']:.2f} \\"
    )
if real:
    lines.append(r"\midrule")
    lines.append(rf"Real paintings & & {real_val['fid_2048']:.1f} & {real['fid_2048']:.1f} & {real['fid']:.3f} & "
                 rf"{real['inception_score']:.2f} & \\")
lines += [r"\bottomrule", r"\end{tabular}"]
(REPORT_DIR / "results_table.tex").write_text("\n".join(lines) + "\n", encoding="utf-8", newline="\n")

chosen_dir = final_dirs[0]  # f-final-seed0: the curves, samples and \Result macros of the report
if not QUICK:  # hw2.report_assets and package_submission.py use runs/hw2/LATEST by default
    (REAL_RUNS_DIR / "LATEST").write_text(chosen_dir.name + "\n", encoding="utf-8")
default_generated = report_assets.GENERATED
report_assets.GENERATED = REPORT_DIR  # the report folder (or runs/hw2-quick/report with QUICK)
try:
    report_assets.export(chosen_dir)
finally:
    report_assets.GENERATED = default_generated
print(f"wrote results_table.tex ({len(report_rows)} rows); exported {chosen_dir.name} -> {REPORT_DIR}")

## 18. Figures for the report

As section 18 of HW1, one figure with no picking (fixed seed 0), from top to bottom: 8 random test paintings; the
terminal baseline and the final model (seed 0, its `best.pt`, EMA weights) from the **same** 8 latent codes $z$;
the final model with the **truncation trick** (Brock et al. 2019: every $|z_i| > 0.7$ drawn again; sharper, less
varied; labeled as such); and for each final sample its **nearest training painting** (pixel MSE over all 60,271):
if a sample were a copy, its nearest painting would be nearly the same image. For scale, the cell also prints the
same distance for 8 validation paintings (real paintings that G never saw). Then the validation FID curves of the
baseline, `t-60ep` and the final model. Both go to `report/generated/`, plus `curves_2x2.pdf`: the final run's training curves as a 2 x 2
panel that fits next to the pictures (`SHOW_TRUNCATION = True` adds the truncation row to the figure).

In [ ]:
import matplotlib.ticker

SHOW_BASE_ROW = False  # True: add the terminal baseline as a row of images.png
SHOW_TRUNCATION = False  # True: add a row with the truncation trick (|z_i| < 0.7) to images.png
final_dir = final_dirs[0]
model_f, weights_f = load_gan(final_dir)
seed_everything(0)
with torch.no_grad():
    z = torch.randn(8, model_f.latent_dim, generator=torch.Generator().manual_seed(0))
    test_set = WikiArt("test", SIZE)
    pick = torch.randperm(len(test_set), generator=torch.Generator().manual_seed(0))[:8]
    figure_rows = [("Test", test_set.images[pick].float() / 255)]
    if SHOW_BASE_ROW and baseline_dir is not None and not QUICK:
        model_b, weights_b = load_gan(baseline_dir)
        figure_rows.append(("Base, no EMA", to_image_range(model_b.generator(z.to(device))).cpu()))
    final_samples = to_image_range(model_f.generator(z.to(device))).cpu()
    figure_rows.append((f"Final ({weights_f.upper()})", final_samples))
    if SHOW_TRUNCATION:
        z_trunc = truncated_z(8, model_f.latent_dim, 0.7, seed=0)
        figure_rows.append(("Final, |z| < 0.7", to_image_range(model_f.generator(z_trunc.to(device))).cpu()))
    index, sample_mse, train_images = nearest_training(final_samples)
    figure_rows.append(("Nearest train", train_images[index].float() / 255))
    val_set = WikiArt("val", SIZE)
    val_pick = torch.randperm(len(val_set), generator=torch.Generator().manual_seed(0))[:8]
    _, val_mse, _ = nearest_training(val_set.images[val_pick].float() / 255)
print(f"pixel MSE to the nearest training painting: samples {sample_mse.mean():.4f}, "
      f"validation paintings {val_mse.mean():.4f} (much lower for the samples = G copies)")

fig, axes = plt.subplots(len(figure_rows), 8, figsize=(7.2, 0.95 * len(figure_rows)), squeeze=False)
for r, (label, images) in enumerate(figure_rows):
    for c in range(8):
        if c < len(images):
            axes[r, c].imshow(images[c].permute(1, 2, 0).clamp(0, 1))
        axes[r, c].set(xticks=[], yticks=[])
    axes[r, 0].set_ylabel(label, rotation=0, ha="right", va="center", fontsize=14)
fig.tight_layout(pad=0.3, w_pad=0.1, h_pad=0.2)
fig.savefig(REPORT_DIR / "images.png", dpi=300)
plt.show()

curve_runs = {"baseline": baseline_dir} if baseline_dir is not None and not QUICK else {}
curve_runs |= {name: d for name in ("l-nonsaturating", "t-60ep") if (d := resolve(name)) is not None}
curve_runs["final (seed 0)"] = final_dir
compare_curves(curve_runs, REPORT_DIR / "fid_curves.pdf")
# The final run's training curves as a 2 x 2 panel, to sit next to images.png in the report.
m = read_csv(final_dir / "metrics.csv")
loss_f = OmegaConf.load(final_dir / "config.yaml").train.loss
has_fid = ~np.isnan(m["val/fid_2048"])
with plt.rc_context({"font.size": 9, "axes.titlesize": 9, "legend.fontsize": 8}):
    fig, axes = plt.subplots(2, 2, figsize=(3.6, 2.5))
    for ax, key, title, ref in ((axes[0, 0], "d_loss", "D loss", BALANCE[loss_f][0]),
                                (axes[0, 1], "g_loss", "G loss", BALANCE[loss_f][1])):
        ax.plot(m["epoch"], m[f"train/{key}"], color="C0")
        ax.axhline(ref, color="gray", ls=":", lw=1.0, label="at D = 1/2")
        ax.set_title(f"{title} ({loss_f})")
        ax.legend(loc="best")
    axes[0, 0].set_ylim(top=max(2.2, 1.1 * BALANCE[loss_f][0]))
    axes[1, 0].plot(m["epoch"], m["train/d_real"], color="C0", label="D(x)")
    axes[1, 0].plot(m["epoch"], m["train/d_fake"], color="C1", ls="--", label="D(G(z))")
    axes[1, 0].set(ylim=(0, 1), title="D outputs", xlabel="epoch")
    axes[1, 0].legend(loc="center", ncol=2, handlelength=1.5, columnspacing=0.8)
    axes[1, 1].plot(m["epoch"][has_fid], m["val/fid_2048"][has_fid], "o-", ms=2.5, color="C2")
    axes[1, 1].set(yscale="log", title="val FID$_{2048}$", xlabel="epoch", yticks=[30, 50, 100, 200])
    axes[1, 1].yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
    axes[1, 1].yaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
    fig.tight_layout(pad=0.4)
    fig.savefig(REPORT_DIR / "curves_2x2.pdf", bbox_inches="tight")
    plt.show()

# The same curves as one strip of 4 panels, to sit under a full-width images.png in the report.
with plt.rc_context({"font.size": 8, "axes.titlesize": 8, "legend.fontsize": 7}):
    fig, axes = plt.subplots(1, 4, figsize=(7.2, 1.75))
    for ax, key, title, ref in ((axes[0], "d_loss", "D loss", BALANCE[loss_f][0]),
                                (axes[1], "g_loss", "G loss", BALANCE[loss_f][1])):
        ax.plot(m["epoch"], m[f"train/{key}"], color="C0")
        ax.axhline(ref, color="gray", ls=":", lw=0.8, label="at D = 1/2")
        ax.set(title=f"{title} ({loss_f}, epoch mean)", xlabel="epoch")
        ax.legend(loc="best")
    axes[2].plot(m["epoch"], m["train/d_real"], color="C0", label="D(x)")
    axes[2].plot(m["epoch"], m["train/d_fake"], color="C1", ls="--", label="D(G(z))")
    axes[2].set(ylim=(0, 1), title="D outputs (epoch mean)", xlabel="epoch")
    axes[2].legend(loc="center right")
    axes[3].plot(m["epoch"][has_fid], m["val/fid_2048"][has_fid], "o-", ms=2.5, color="C2")
    axes[3].set(yscale="log", title="val FID$_{2048}$ (n = 2,000)", xlabel="epoch", yticks=[30, 50, 100, 200])
    axes[3].yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
    axes[3].yaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
    fig.tight_layout(pad=0.4)
    fig.savefig(REPORT_DIR / "curves_strip.pdf", bbox_inches="tight")
    plt.show()
print(f"wrote images.png, fid_curves.pdf, curves_2x2.pdf and curves_strip.pdf -> {REPORT_DIR}")